## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-06-59-00 +0000,wsj,Demand for Chinese Goods Is Soaring but Econom...,https://www.wsj.com/economy/chinas-domestic-de...
1,2026-11-16-06-16-00 +0000,wsj,Asian Equities Drop as U.S. Chip Selloff Spill...,https://www.wsj.com/finance/stocks/asian-equit...
2,2026-11-16-06-14-00 +0000,wsj,U.S. 10-Year Treasurys Seen Attractive to Buy ...,https://www.wsj.com/finance/-63dc4bbf?mod=rss_...
3,2026-11-16-06-04-24 +0000,bbc,Workers should get paid time off to train for ...,https://www.bbc.co.uk/news/articles/cq8j4yn1vj...
4,2026-11-16-06-03-39 +0000,bbc,Thrift shop ask donors to nominate a charity,https://www.bbc.co.uk/news/articles/c804wp1y2k...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2751/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
183,trump,19
107,new,13
224,nvidia,11
66,ground,11
35,huang,10


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
137,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...,64
151,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...,63
227,2026-11-15-09-23-51 +0000,nypost,‘SNL’ roasts ex-Nvidia boss Jensen Huang over ...,https://nypost.com/2026/11/15/business/snl-roa...,60
113,2026-11-15-20-20-57 +0000,nypost,Jensen Huang eyeing comeback as Nvidia CEO for...,https://nypost.com/2026/11/15/business/jensen-...,55
154,2026-11-15-17-10-54 +0000,nypost,Trump blames Biden admin for FAA outage debacl...,https://nypost.com/2026/11/15/us-news/trump-bl...,55


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
137,64,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...
151,63,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...
63,37,2026-11-15-23-35-02 +0000,cbc,'I cheated': South Korean admits he used AI to...,https://www.cbc.ca/news/world/south-korea-tax-...
204,32,2026-11-15-11-00-00 +0000,latimes,Offshore oil rigs could be 'toppled' off Calif...,https://www.latimes.com/environment/story/2026...
210,31,2026-11-15-10-30-00 +0000,wsj,High mortgage rates have crushed homeowners’ b...,https://www.wsj.com/economy/housing/theyre-luc...
227,31,2026-11-15-09-23-51 +0000,nypost,‘SNL’ roasts ex-Nvidia boss Jensen Huang over ...,https://nypost.com/2026/11/15/business/snl-roa...
206,30,2026-11-15-10-34-22 +0000,nypost,Vagrant sets up City Hall encampment along Mam...,https://nypost.com/2026/11/15/us-news/mamdanis...
125,29,2026-11-15-19-30-51 +0000,chitri,Israel kills a top Hamas commander as Jordan s...,https://www.chicagotribune.com/2026/11/15/isra...
150,28,2026-11-15-17-27-57 +0000,nypost,Dem Sen. Chris Van Hollen boasts he has ‘very ...,https://nypost.com/2026/11/15/us-news/van-holl...
83,27,2026-11-15-22-13-00 +0000,wsj,"Miguel Ángel Félix Gallardo, Architect of Mexi...",https://www.wsj.com/world/americas/miguel-ange...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
